# 🕵️ Dia 08 — TLS e certificados: o que o investigador lê num certificado

**Investigador Cyber 90 Dias** · Fase 1 · Rank Jutsu `D`

> Baseado no *Day 8* de [cyber-investigator-skills](https://github.com/thiefdroppers/cyber-investigator-skills/blob/main/curriculum/phase-1-foundations/day-08.md), © ThiefDroppers (MIT).
> Notebook, camada BR e Camada 8 por Deivison Lourenço ([@Ridd1kulusC0d3r](https://github.com/Ridd1kulusC0d3r)).

---

### 🎬 Onde paramos

Terça, 09h10. Marta: *"Um cooperado diz que o site tinha cadeado. Então era seguro?"* Rafa: *"O cadeado diz que a conversa é cifrada. Não diz com quem."*

O certificado do `serraazul-pix.example` começou a valer na véspera do disparo, e a lista de nomes traz `serraazul-seguranca.example`. No CT, um nome novo: `coopserra-azul.example`.

Neste notebook você vai:

1. **Ler um certificado sem `openssl`**: base64 → DER → ASN.1, só com a biblioteca padrão.
2. **Calcular a impressão digital** e ver por que ela difere do hash do arquivo `.pem`.
3. **Obter um certificado real** com `ssl` (com plano B offline).
4. **Processar o JSON do crt.sh**, deduplicar por número de série e gerar o Gantt do IscaKit.
5. **Jogar o quiz "o cadeado prova isso?"** e gerar a ficha de perfil.

🐍 Só biblioteca padrão do Python. Nada para instalar. Rode as células de cima para baixo (Shift+Enter). Funciona sem internet: toda célula que tenta a rede tem plano B embutido.

In [ ]:
# ------------------------------------------------------------
# DADOS DO CASO (Operação Isca — tudo fictício)
# Primeiro tentamos baixar do repositório no GitHub; se não houver
# internet (ou o repo estiver fora do ar), usamos a cópia embutida.
# ------------------------------------------------------------
import urllib.request   # baixa arquivos pela web, sem instalar nada

BASE = "https://raw.githubusercontent.com/Ridd1kulusC0d3r/investigador-cyber-90-dias/main/caso-continuo/dados/dia-08/"

# Cópias embutidas: o plano B. São idênticas aos arquivos do repo.
EMBUTIDO = {}
EMBUTIDO['serraazul-pix.example.pem'] = '-----BEGIN CERTIFICATE-----\nMIIB9zCCAZygAwIBAgISBKfD4Zsi8NboHFp3MLnU4vYaMAoGCCqGSM49BAMCMFcx\nCzAJBgNVBAYTAkJSMScwJQYDVQQKDB5BQyBHcmF0dWl0YSBFeGVtcGxvIChmaWN0\naWNpYSkxHzAdBgNVBAMMFkFDIEdyYXR1aXRhIEV4ZW1wbG8gUjcwHhcNMjYxMDA0\nMjE0NzEyWhcNMjcwMTAyMjE0NzExWjAgMR4wHAYDVQQDDBVzZXJyYWF6dWwtcGl4\nLmV4YW1wbGUwWTATBgcqhkjOPQIBBggqhkjOPQMBBwNCAARkNcTnEHUmefERnEe3\nCdFq6u1IEeTwSymklP+S2tEUpYDA5kC36tKYm96dG0r+PvORn7LeRVN1CK46VtKP\nR7mmo38wfTBYBgNVHREEUTBPghVzZXJyYWF6dWwtcGl4LmV4YW1wbGWCGXd3dy5z\nZXJyYWF6dWwtcGl4LmV4YW1wbGWCG3NlcnJhYXp1bC1zZWd1cmFuY2EuZXhhbXBs\nZTATBgNVHSAEDDAKMAgGBmeBDAECATAMBgNVHRMBAf8EAjAAMAoGCCqGSM49BAMC\nA0kAMEYCIQDNVHP9KJ/txwNYJxkEibyt9QQR2Jz2frnUenPXMylgGwIhAJmQBKvd\nzkefmKTA7LcJmc2NREVOYZlKT7bNaBMBIpT1\n-----END CERTIFICATE-----\n'
EMBUTIDO['crtsh.json'] = '[\n {\n  "id": 90000000101,\n  "issuer_name": "C=BR, O=AC Gratuita Exemplo (ficticia), CN=AC Gratuita Exemplo R7",\n  "common_name": "serraazul-pix.example",\n  "name_value": "serraazul-pix.example\\nwww.serraazul-pix.example\\nserraazul-seguranca.example",\n  "entry_timestamp": "2026-10-04T21:47:13.402",\n  "not_before": "2026-10-04T21:47:12",\n  "not_after": "2027-01-02T21:47:11",\n  "serial_number": "04a7c3e19b22f0d6e81c5a7730b9d4e2f61a"\n },\n {\n  "id": 90000000102,\n  "issuer_name": "C=BR, O=AC Gratuita Exemplo (ficticia), CN=AC Gratuita Exemplo R7",\n  "common_name": "serraazul-pix.example",\n  "name_value": "serraazul-pix.example\\nwww.serraazul-pix.example\\nserraazul-seguranca.example",\n  "entry_timestamp": "2026-10-04T21:47:12.990",\n  "not_before": "2026-10-04T21:47:12",\n  "not_after": "2027-01-02T21:47:11",\n  "serial_number": "04a7c3e19b22f0d6e81c5a7730b9d4e2f61a"\n },\n {\n  "id": 90000000377,\n  "issuer_name": "C=BR, O=AC Gratuita Exemplo (ficticia), CN=AC Gratuita Exemplo R7",\n  "common_name": "coopserra-azul.example",\n  "name_value": "coopserra-azul.example\\nwww.coopserra-azul.example",\n  "entry_timestamp": "2026-10-06T02:13:40.118",\n  "not_before": "2026-10-06T02:13:39",\n  "not_after": "2027-01-04T02:13:38",\n  "serial_number": "03bb19e0c7d2a4f5e6a1b2c3d4e5f6a7b8c9"\n },\n {\n  "id": 90000000012,\n  "issuer_name": "C=BR, O=AC Exemplo OV (ficticia), CN=AC Exemplo OV 2",\n  "common_name": "serraazul.example",\n  "name_value": "serraazul.example\\nwww.serraazul.example\\nib.serraazul.example",\n  "entry_timestamp": "2025-11-03T13:02:11.000",\n  "not_before": "2025-11-03T13:02:10",\n  "not_after": "2026-11-03T13:02:09",\n  "serial_number": "5a1e0f2d9c3b4a5968778695a4b3c2d1e0f1"\n }\n]\n'

def carregar(nome):
    """Devolve o texto do arquivo: da rede se der, senão da cópia embutida."""
    try:
        with urllib.request.urlopen(BASE + nome, timeout=5) as r:   # timeout: não travar a aula
            texto = r.read().decode("utf-8")
        if texto == EMBUTIDO[nome]:                                   # confere com a cópia local
            print(f"🌐 {nome}: baixado do GitHub (idêntico à cópia embutida)")
            return texto
        print(f"⚠️ {nome}: a versão do GitHub difere da embutida; usando a embutida (é a que os hashes do notebook esperam)")
        return EMBUTIDO[nome]
    except Exception as erro:                                         # sem rede, 404, proxy...
        print(f"📦 {nome}: usando a cópia embutida ({type(erro).__name__})")
        return EMBUTIDO[nome]

print(f"{len(EMBUTIDO)} arquivo(s) do caso disponíveis:", ", ".join(EMBUTIDO))

In [ ]:
import hashlib, base64, re

TEXTOS = {nome: carregar(nome) for nome in EMBUTIDO}
PEM = TEXTOS["serraazul-pix.example.pem"]

# PEM = base64 entre as linhas BEGIN/END. Dentro, os bytes DER do certificado.
miolo = re.search(r"-----BEGIN CERTIFICATE-----(.+?)-----END CERTIFICATE-----", PEM, re.S).group(1)
DER = base64.b64decode("".join(miolo.split()))          # tira quebras de linha e decodifica

print("SHA-256 do ARQUIVO .pem (texto):  ", hashlib.sha256(PEM.encode()).hexdigest())
fp = hashlib.sha256(DER).hexdigest().upper()
print("Impressão digital SHA-256 (DER):  ", ":".join(fp[i:i + 2] for i in range(0, 64, 2)))
print("\n➡️ O hash do arquivo protege a SUA cópia (custódia). A impressão digital identifica O CERTIFICADO;")
print("   é ela que você cita no relatório e que bate com o 'sha256 Fingerprint' do openssl.")

## 1. Um mini-leitor de DER (ASN.1)

Um certificado X.509 é uma árvore de blocos **TLV**: *Tag* (1 byte: que tipo de coisa), *Length* (quantos bytes) e *Value*. Blocos do tipo `SEQUENCE` (`0x30`) contêm outros blocos. Com isso e um pouco de `int.from_bytes`, dá para ler tudo o que o investigador precisa.

In [ ]:
def tlv(dados, pos=0):
    """Lê UM bloco TLV a partir de pos. Devolve (tag, valor_em_bytes, posição_seguinte)."""
    tag = dados[pos]
    tam = dados[pos + 1]
    pos += 2
    if tam & 0x80:                               # bit alto ligado = tamanho "longo"
        n = tam & 0x7F                           # quantos bytes guardam o tamanho
        tam = int.from_bytes(dados[pos:pos + n], "big")
        pos += n
    return tag, dados[pos:pos + tam], pos + tam

def filhos(valor):
    """Quebra o conteúdo de uma SEQUENCE/SET na lista de blocos internos."""
    itens, pos = [], 0
    while pos < len(valor):
        tag, v, pos = tlv(valor, pos)
        itens.append((tag, v))
    return itens

def oid(b):
    """Decodifica um OBJECT IDENTIFIER (ex.: 2.23.140.1.2.1)."""
    partes = [b[0] // 40, b[0] % 40]
    n = 0
    for byte in b[1:]:
        n = (n << 7) | (byte & 0x7F)             # 7 bits por byte
        if not byte & 0x80:                      # último byte do número
            partes.append(n); n = 0
    return ".".join(map(str, partes))

NOMES_OID = {"2.5.4.3": "CN", "2.5.4.6": "C", "2.5.4.10": "O",
             "2.5.29.17": "subjectAltName", "2.5.29.32": "certificatePolicies", "2.5.29.19": "basicConstraints",
             "2.23.140.1.2.1": "DV (validado por domínio)", "2.23.140.1.2.2": "OV (validado por organização)",
             "2.23.140.1.1": "EV (validação estendida)"}

def nome_x500(valor):
    """Name = SEQUENCE de SET de (OID, texto) → 'C=BR, O=..., CN=...'."""
    partes = []
    for _, conjunto in filhos(valor):
        for _, par in filhos(conjunto):
            (_, o), (_, texto) = filhos(par)
            partes.append(f"{NOMES_OID.get(oid(o), oid(o))}={texto.decode()}")
    return ", ".join(partes)

from datetime import datetime, timezone

def hora(tag, valor):
    """UTCTime (0x17, AAMMDDhhmmssZ) ou GeneralizedTime (0x18, AAAAMMDDhhmmssZ) → datetime UTC."""
    fmt = "%y%m%d%H%M%SZ" if tag == 0x17 else "%Y%m%d%H%M%SZ"
    return datetime.strptime(valor.decode(), fmt).replace(tzinfo=timezone.utc)

def ler_certificado(der):
    _, cert, _ = tlv(der)                        # Certificate ::= SEQUENCE
    tbs = filhos(cert)[0][1]                     # 1º filho: tbsCertificate (o "corpo")
    campos = filhos(tbs)
    i = 1 if campos[0][0] == 0xA0 else 0         # [0] versão é opcional
    sb = campos[i][1]
    serial = (sb[1:] if len(sb) > 1 and sb[0] == 0 else sb).hex().upper()   # tira só o byte de sinal 00, como o openssl
    emissor = nome_x500(campos[i + 2][1])
    nb_tag, nb = filhos(campos[i + 3][1])[0]
    na_tag, na = filhos(campos[i + 3][1])[1]
    sujeito = nome_x500(campos[i + 4][1])
    info = {"serial": serial, "emissor": emissor, "sujeito": sujeito,
            "notBefore": hora(nb_tag, nb), "notAfter": hora(na_tag, na), "SAN": [], "políticas": []}
    for tag, valor in campos[i + 6:]:
        if tag != 0xA3:                          # [3] extensões
            continue
        for _, ext in filhos(filhos(valor)[0][1]):
            partes = filhos(ext)
            nome = oid(partes[0][1])
            conteudo = partes[-1][1]             # OCTET STRING com outro DER dentro
            _, interno, _ = tlv(conteudo)
            if nome == "2.5.29.17":              # SAN: [2] dNSName
                info["SAN"] = [v.decode() for t, v in filhos(interno) if t == 0x82]
            if nome == "2.5.29.32":              # políticas: SEQUENCE de PolicyInformation
                info["políticas"] = [oid(filhos(p)[0][1]) for _, p in filhos(interno)]
    return info

CERT = ler_certificado(DER)
for k, v in CERT.items():
    print(f"{k:11} {v}")
print("\nNível de validação:", [NOMES_OID.get(p, p) for p in CERT["políticas"]])

> Compare com o `openssl` (se tiver instalado): `openssl x509 -in serraazul-pix.example.pem -noout -subject -issuer -dates -serial -fingerprint -sha256 -ext subjectAltName,certificatePolicies`. Os valores precisam bater.

## 2. Âncora na linha do tempo

In [ ]:
from datetime import timedelta

BRT = timezone(timedelta(hours=-3), "BRT")
primeiro_sms = datetime(2026, 10, 5, 7, 12, tzinfo=BRT)          # fato fixo do caso

nb = CERT["notBefore"]
vida = (CERT["notAfter"] - nb).total_seconds() / 86400
print(f"notBefore: {nb:%Y-%m-%d %H:%M:%S} UTC = {nb.astimezone(BRT):%Y-%m-%d %H:%M} BRT")
print(f"Vida útil: {vida:.0f} dias (típico de AC gratuita automatizada)")
print(f"Começou a valer {(primeiro_sms - nb).total_seconds() / 3600:.1f} h antes do primeiro SMS.")

# A lista SAN amarra domínios. Algum deles já apareceu no caso?
JA_VISTOS = {"serraazul-seguranca.example": "Dia 3 (SYN sem resposta) e Dia 6 (mesmo token TXT)"}
for nome in CERT["SAN"]:
    if nome in JA_VISTOS:
        print(f"🔗 SAN {nome} → já visto no {JA_VISTOS[nome]}. Segunda fonte independente para o vínculo.")

## 3. Um certificado de verdade (com plano B)

O módulo `ssl` busca o certificado de um servidor real. Sem rede, o notebook avisa e segue.

In [ ]:
#@title Certificado real { display-mode: "form" }
HOST = "www.wikipedia.org"  #@param {type:"string"}

import ssl, socket
try:
    ctx = ssl.create_default_context()
    with socket.create_connection((HOST, 443), timeout=6) as s:
        with ctx.wrap_socket(s, server_hostname=HOST) as t:      # server_hostname = o SNI
            der_real = t.getpeercert(binary_form=True)
            versao = t.version()
    real = ler_certificado(der_real)
    print(f"Conectado a {HOST} ({versao}) em {datetime.now(timezone.utc):%Y-%m-%dT%H:%M:%SZ}")
    print("Emissor:", real["emissor"])
    print("Validade:", real["notBefore"], "→", real["notAfter"])
    print(f"SAN: {len(real['SAN'])} nomes, ex.: {real['SAN'][:4]}")
    print("Política:", [NOMES_OID.get(p, p) for p in real["políticas"]])
    print("Impressão digital:", hashlib.sha256(der_real).hexdigest())
except Exception as erro:
    print(f"Sem rede ou bloqueado ({type(erro).__name__}). Use o certificado do caso acima;")
    print("no seu computador, rode o openssl do Passo 1 do README.")

## 4. O crt.sh e a linha do tempo do IscaKit

O JSON do crt.sh traz **pré-certificado e certificado final** com o mesmo número de série. Deduplicamos por `serial_number` (o `unique_by` do `jq` no original) e montamos o Gantt.

In [ ]:
import json

registros = json.loads(TEXTOS["crtsh.json"])
print(len(registros), "linhas no crt.sh")

por_serial = {}
for r in sorted(registros, key=lambda r: r["entry_timestamp"]):
    por_serial.setdefault(r["serial_number"], r)          # fica o 1º registro de cada serial
unicos = sorted(por_serial.values(), key=lambda r: r["not_before"])
print(len(unicos), "certificados únicos (por número de série)\n")

vistos, eventos = set(), []
for r in unicos:
    nomes = r["name_value"].split("\n")
    novos = [n for n in nomes if n not in vistos]
    vistos.update(nomes)
    cn_emissor = re.search(r"CN=([^,]+)", r["issuer_name"]).group(1)
    print(f"{r['not_before']}  id {r['id']}  {cn_emissor:24} {', '.join(nomes)}")
    if r["not_before"] > "2026-10-05T10:12:00":           # depois do primeiro SMS (07h12 BRT)
        eventos.append(f"Nome visto pela 1ª vez DEPOIS do disparo: {', '.join(novos)} (registrado no CT em {r['entry_timestamp']} UTC)")
print()
print("\n".join(eventos))

gantt = ["gantt", "    title Certificados do caso Operação Isca (crt.sh, deduplicado por serial)",
         "    dateFormat YYYY-MM-DD", "    axisFormat %Y-%m"]
for secao, filtro in [("Cooperativa (legítimo)", lambda r: r["common_name"] == "serraazul.example"),
                      ("IscaKit", lambda r: r["common_name"] != "serraazul.example")]:
    gantt.append(f"    section {secao}")
    for r in filter(filtro, unicos):
        cn_emissor = re.search(r"CN=([^,]+)", r["issuer_name"]).group(1)
        gantt.append(f"    {r['common_name']}, {cn_emissor}, id {r['id']} :{r['not_before'][:10]}, {r['not_after'][:10]}")
GANTT = "\n".join(gantt)
print("\n" + GANTT)

## 5. 🎮 Quiz: o cadeado prova isso?

**S** (prova) ou **N** (não prova). `MODO_JOGO = False` mostra só as respostas.

In [ ]:
QUIZ = [
    ("A conexão entre o celular da vítima e 203.0.113.77 estava cifrada.", True,
     "É exatamente o que o TLS garante."),
    ("O site pertence à Cooperativa Serra Azul.", False,
     "Certificado DV prova controle do DOMÍNIO serraazul-pix.example, não identidade."),
    ("Quem controlava serraazul-pix.example em 4/out também controlava serraazul-seguranca.example.", True,
     "Para emitir um DV com os dois nomes, a AC validou o controle dos dois. (Prova sobre domínios, não sobre pessoas.)"),
    ("O site foi montado por volta de 4/out, 21h47 UTC.", True,
     "notBefore costuma ficar a minutos da emissão: âncora forte (ainda é inferência sobre 'montar o site')."),
    ("Um certificado de AC gratuita é sinal de site malicioso.", False,
     "ACs gratuitas emitem para sites legítimos e maliciosos. O emissor sozinho não prova nada."),
    ("O certificado tem validade jurídica de 'certificado digital' ICP-Brasil.", False,
     "Certificado TLS de site não é certificado ICP-Brasil (MP 2.200-2/2001)."),
    ("Um sensor passivo na rede da cooperativa, numa sessão TLS 1.3, registraria o certificado.", False,
     "No TLS 1.3 o certificado vai cifrado; o sensor vê o SNI."),
    ("O CAA de serraazul.example deveria ter impedido este certificado.", False,
     "O CAA vale para o domínio que o publica. serraazul-pix.example tem DNS próprio, sem CAA."),
]
MODO_JOGO = True

acertos = 0
for i, (frase, prova, porque) in enumerate(QUIZ, start=1):
    print(f"\n[{i}/{len(QUIZ)}] {frase}")
    if MODO_JOGO:
        r = ""
        while r not in ("S", "N"):
            r = input("   O certificado prova isso? (S/N): ").strip().upper()
        ok = (r == "S") == prova
        acertos += ok
        print("   ✅ Acertou!" if ok else "   ❌ Não foi dessa vez.")
    print(f"   Resposta: {'prova' if prova else 'NÃO prova'} — {porque}")
if MODO_JOGO:
    print(f"\n🏁 {acertos}/{len(QUIZ)}", "— Checkpoint BR cumprido. 🥷" if acertos >= 6 else "— meta: 6 de 8.")

## 6. 📝 A ficha de perfil (Passo 5) e a linha do tempo

In [ ]:
from pathlib import Path

primeiro = unicos[0]
ficha = f"""| Campo | Valor | Nota investigativa |
|---|---|---|
| Host consultado (SNI) | serraazul-pix.example | Coleta da VM vigia-lab01 via VPN de pesquisa |
| Coletado (UTC) / a partir do IP | (sua hora de coleta) / 203.0.113.77 | Mesmo IP do Dia 2 |
| CN do Subject | {CERT['sujeito']} | |
| Quantidade de SAN / SANs notáveis | {len(CERT['SAN'])} / {', '.join(CERT['SAN'])} | serraazul-seguranca.example liga ao Dia 3 e ao token do Dia 6 |
| Emissor | {CERT['emissor']} | AC gratuita (fictícia); emissor sozinho não prova nada |
| Nível de validação (OID da política) | {', '.join(CERT['políticas'])} | {', '.join(NOMES_OID.get(p, p) for p in CERT['políticas'])} |
| notBefore / notAfter / vida útil em dias | {CERT['notBefore']:%Y-%m-%dT%H:%M:%SZ} / {CERT['notAfter']:%Y-%m-%dT%H:%M:%SZ} / {vida:.0f} | {(primeiro_sms - nb).total_seconds() / 3600:.1f} h antes do 1º SMS |
| Número de série | {CERT['serial']} | Citar no pedido de revogação |
| Impressão digital SHA-256 | {hashlib.sha256(DER).hexdigest()} | Identificador do certificado no relatório |
| Bate com o CAA do DNS (Dia 6)? | Domínio do golpe não publica CAA | O CAA de serraazul.example não se aplica a este domínio |
| Primeiro certificado já registrado para este domínio (crt.sh) | id {primeiro['id']}, {primeiro['not_before']} | Domínio novo: nenhum histórico anterior |
| Cadência de renovação observada | nenhuma ainda | Domínio com 1 dia de vida |
"""
resposta_marta = ("O cadeado só mostra que a conversa entre o celular e o site estava cifrada. "
                  "Ele não diz quem é o dono do site: qualquer pessoa consegue um cadeado de graça em minutos. "
                  "A Serra Azul só usa serraazul.example; qualquer outro endereço, mesmo com cadeado, não é nosso.")
Path("notas").mkdir(exist_ok=True)
Path("notas/day08-cert-card-isca.md").write_text(ficha + "\n**Para a Marta:** " + resposta_marta + "\n", encoding="utf-8")
Path("notas/day08-timeline-isca.md").write_text("```mermaid\n" + GANTT + "\n```\n", encoding="utf-8")
print(ficha)
print("Para a Marta:", resposta_marta)
print("\n💾 notas/day08-cert-card-isca.md · notas/day08-timeline-isca.md")
try:
    from google.colab import files
    files.download("notas/day08-cert-card-isca.md"); files.download("notas/day08-timeline-isca.md")
except ImportError:
    print("(Fora do Colab: os arquivos ficaram na pasta notas/)")

## ✅ Antes de fechar o Dia 8

- [ ] Leu o certificado sem `openssl` e conferiu com ele (se tiver)
- [ ] Sabe a diferença entre hash do `.pem` e impressão digital do certificado
- [ ] Vínculo `serraazul-pix` ↔ `serraazul-seguranca` com duas fontes independentes
- [ ] `coopserra-azul.example` marcado como nome novo depois do disparo
- [ ] Quiz com pelo menos 6/8
- [ ] Fez também a prática original (`www.wikipedia.org`, SNI no Wireshark, crt.sh de 12 meses)

**Amanhã (Dia 9):** o firewall da sede. Por que as conexões perto do meio-dia deram `S0`? E quem é `10.20.40.57`, que não é da sede?

---
<sub>Baseado em cyber-investigator-skills por ThiefDroppers (MIT) · Investigador Cyber 90 Dias por Deivison Lourenço</sub>